# Linguistic Capital or Linguistic Barrier?
## Comparative analysis of accent, identity, and socio-economic stratification

Reproducible analysis of the supplied 60-record participant dataset. Locations are derived from participant IDs (`S-` = Sandton; `SW-` = Soweto). The label order is fixed as Standard, Inclusion, Neutral. This notebook reproduces the contingency table and inference, computes diversity indices, and creates three publication-oriented figures.

**Reliability limitation:** the source contains one final label per participant, not paired independent coder ratings. Cohen’s kappa below is therefore a self-agreement implementation check plus a shuffled-label baseline, not inter-rater reliability.

### 1. Imports and data loading

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, fisher_exact

DATA_PATH = Path("interview_dataset_sandton_soweto.csv")
LABELS = ["Standard", "Inclusion", "Neutral"]
LOCATIONS = ["Sandton", "Soweto"]
df = pd.read_csv(DATA_PATH)
required = {"Participant ID", "Raw Response Extract", "Code", "Category", "Final Label"}
assert required.issubset(df.columns)
df["Location"] = np.where(df["Participant ID"].astype(str).str.startswith("SW-"), "Soweto", "Sandton")
df["Final Label"] = df["Final Label"].astype(str).str.strip()
assert len(df) == 60 and not df["Participant ID"].duplicated().any()
assert set(df["Location"]) == set(LOCATIONS) and set(df["Final Label"]) == set(LABELS)
df.head()

### 2. Cleaning checks and cross-tabulation

In [ ]:
print("Rows:", len(df))
print("Missing values:\n", df.isna().sum())
print("Location counts:\n", df["Location"].value_counts())
print("Labels:\n", df["Final Label"].value_counts())
contingency = pd.crosstab(df["Location"], df["Final Label"]).reindex(index=LOCATIONS, columns=LABELS, fill_value=0)
contingency

### 3. Hypothesis testing and effect size
Pearson’s chi-square test uses no continuity correction. Four of the six expected cell counts are below 5, so the asymptotic p-value should be interpreted cautiously; the companion R script runs the 2×3 Fisher–Freeman–Halton exact test.

In [ ]:
chi2, p_value, dof, expected = chi2_contingency(contingency, correction=False)
n = contingency.to_numpy().sum()
cramers_v = np.sqrt(chi2 / (n * min(contingency.shape[0]-1, contingency.shape[1]-1)))
print("Expected counts:\n", pd.DataFrame(expected, index=LOCATIONS, columns=LABELS))
print(f"Pearson chi-square = {chi2:.3f}; df = {dof}; p = {p_value:.12g}; Cramer's V = {cramers_v:.3f}")
assert np.isclose(chi2, 49.0909090909) and dof == 2 and np.isclose(cramers_v, .904534)

### 4. Agreement check: Cohen’s kappa
Because the dataset has no second coder, only self-agreement (expected κ=1) and a fixed-seed permutation benchmark can be demonstrated. Neither should be described as empirical inter-rater reliability.

In [ ]:
def cohens_kappa(a, b):
    a, b = np.asarray(a), np.asarray(b)
    observed = np.mean(a == b)
    categories = np.union1d(a, b)
    expected = sum(np.mean(a == c) * np.mean(b == c) for c in categories)
    return 1.0 if np.isclose(observed, 1) and np.isclose(expected, 1) else (observed-expected)/(1-expected)
labels = df["Final Label"].to_numpy()
print("Self-agreement kappa (implementation check only):", cohens_kappa(labels, labels))
rng = np.random.default_rng(20250308)
permuted = np.array([cohens_kappa(labels, rng.permutation(labels)) for _ in range(2000)])
print("Shuffled baseline mean and 95% quantiles:", permuted.mean(), np.quantile(permuted, [.025,.975]))

### 5. Shannon entropy and Simpson diversity

In [ ]:
def diversity(counts):
    p = np.asarray(counts, dtype=float); p = p[p > 0] / p.sum()
    return pd.Series({"Shannon (nats)": -np.sum(p*np.log(p)),
                      "Shannon (bits)": -np.sum(p*np.log2(p)),
                      "Simpson diversity (1−Σp²)": 1-np.sum(p**2),
                      "Simpson concentration (Σp²)": np.sum(p**2)})
diversity_table = pd.DataFrame({site: diversity(contingency.loc[site].values) for site in LOCATIONS}).T
diversity_table

### 6. Figure 1 — Ideological distribution

In [ ]:
sns.set_theme(style="whitegrid", context="notebook", font_scale=1.05)
PALETTE = {"Standard":"#355C7D", "Inclusion":"#2A9D8F", "Neutral":"#E9C46A"}
plt.rcParams.update({"axes.spines.top":False, "axes.spines.right":False, "figure.dpi":120})
long = contingency.rename_axis("Location").reset_index().melt(id_vars="Location", var_name="Final label", value_name="Participants")
fig, ax = plt.subplots(figsize=(8,5))
sns.barplot(data=long, x="Location", y="Participants", hue="Final label", hue_order=LABELS, palette=PALETTE, ax=ax, edgecolor="white")
ax.set(title="Figure 1. Ideological distribution by study site", xlabel="", ylabel="Participants (n)")
ax.legend(title="Final label", frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(.5,1.0))
for container in ax.containers: ax.bar_label(container, fmt="%d", padding=2)
fig.tight_layout(); plt.show()

### 7. Figure 2 — Thematic dimensions (source categories)

In [ ]:
themes = pd.crosstab(df["Location"], df["Category"]).reindex(index=LOCATIONS, fill_value=0)
themes_long = themes.rename_axis("Location").reset_index().melt(id_vars="Location", var_name="Thematic dimension", value_name="Participants")
fig, ax = plt.subplots(figsize=(9,5))
sns.barplot(data=themes_long, x="Thematic dimension", y="Participants", hue="Location", palette={"Sandton":"#355C7D", "Soweto":"#D1495B"}, ax=ax, edgecolor="white")
ax.set(title="Figure 2. Thematic dimensions by study site", xlabel="", ylabel="Participants (n)")
ax.tick_params(axis="x", rotation=15); ax.legend(title="", frameon=False)
for container in ax.containers: ax.bar_label(container, fmt="%d", padding=2)
fig.tight_layout(); plt.show()

### 8. Figure 3 — Theoretical framework
This is a conceptual organizing diagram, not an estimated causal model.

In [ ]:
fig, ax = plt.subplots(figsize=(10,4.8)); ax.set_xlim(0,10); ax.set_ylim(0,5); ax.axis("off")
boxes = [(0.3,1.7,2.5,1.5,"Socio-economic\ncontext\nSandton / Soweto","#E7EEF5"),
         (3.75,1.7,2.5,1.5,"Accent, identity &\nlinguistic resources","#E5F3F0"),
         (7.2,1.7,2.5,1.5,"Ideological positioning\nStandard / Inclusion / Neutral","#FFF1D6")]
for x,y,w,h,label,color in boxes:
    ax.add_patch(plt.Rectangle((x,y),w,h,facecolor=color,edgecolor="#34495E",linewidth=1.5))
    ax.text(x+w/2,y+h/2,label,ha="center",va="center",fontsize=11)
ax.annotate("",xy=(3.65,2.45),xytext=(2.9,2.45),arrowprops=dict(arrowstyle="->",lw=1.8,color="#34495E"))
ax.annotate("",xy=(7.1,2.45),xytext=(6.35,2.45),arrowprops=dict(arrowstyle="->",lw=1.8,color="#34495E"))
ax.text(5,4.25,"Figure 3. Conceptual framework for comparative interpretation",ha="center",fontsize=13,weight="bold")
ax.text(5,.75,"Interpretive framework; arrows do not imply causal identification.",ha="center",fontsize=9,color="#555")
fig.tight_layout(); plt.show()

### 9. Export tables and figures (optional)
Uncomment the cell below to save the result tables and publication-resolution PNG/PDF figures in `analysis_outputs/`.

In [ ]:
# OUT = Path("analysis_outputs"); OUT.mkdir(exist_ok=True)
# contingency.to_csv(OUT/"contingency_table.csv")
# diversity_table.to_csv(OUT/"diversity_indices.csv")
# (fig1, ax1) = plt.subplots(figsize=(7.2,4.8))  # rerun a figure cell and save with fig.savefig(...)